# LangChain Agent Class Demo: Search + Weather + RAG

This Colab project demonstrates a **tool-using LangChain agent** that can decide between:

1. **External Search Tool** — current web information using SerpAPI  
2. **Weather Tool** — live weather using OpenWeatherMap API  
3. **RAG Tool** — answers from your uploaded PDF using FAISS vector search

The idea is simple:

> A normal chatbot only answers from model knowledge.  
> An agent can choose tools: search the web, call APIs, or retrieve from your private document.

---

## Class Demo Scenario

Ask the agent questions like:

- `What is the latest news about ISRO?`
- `What is the weather in Kolkata?`
- `According to the uploaded PDF, what are the candidate's main skills?`
- `Compare the weather in Bengaluru with current news about traffic there.`

The agent decides which tool to call.


In [1]:
# ============================================================
# 1. Install required packages
# ============================================================

!pip install -q   langchain   langchain-community   langchain-openrouter   langchain-huggingface   sentence-transformers   faiss-cpu   pypdf   google-search-results   requests


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confli

## 2. Add API keys safely

You need three keys:

- `OPENROUTER_API_KEY` — for the LLM
- `SERPAPI_API_KEY` — for Google search through SerpAPI
- `OPENWEATHER_API_KEY` — for weather data

In class, explain that keys should not be hard-coded in notebooks. Use `getpass` instead.


In [2]:
# ============================================================
# 2. API keys
# ============================================================

import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OPENROUTER_API_KEY: ")
os.environ["SERPAPI_API_KEY"] = getpass.getpass("Enter SERPAPI_API_KEY: ")
os.environ["OPENWEATHER_API_KEY"] = getpass.getpass("Enter OPENWEATHER_API_KEY: ")

print("✅ API keys loaded into environment variables")


Enter OPENROUTER_API_KEY: ··········
Enter SERPAPI_API_KEY: ··········
Enter OPENWEATHER_API_KEY: ··········
✅ API keys loaded into environment variables


## 3. Import libraries

This uses the newer LangChain `create_agent` interface plus standard `@tool` functions.


In [3]:
# ============================================================
# 3. Imports
# ============================================================

import os
import requests

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openrouter import ChatOpenRouter

from langchain_community.utilities import SerpAPIWrapper
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✅ Imports complete")


/tmp/ipykernel_1089/31362068.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SerpAPIWrapper


✅ Imports complete


## 4. Upload a PDF for RAG

For demo, upload any PDF such as:

- Resume
- HR policy
- Product document
- Company FAQ
- Course notes

The RAG tool will answer only from this uploaded document.


In [4]:
# ============================================================
# 4. Upload PDF in Colab
# ============================================================

from google.colab import files

uploaded = files.upload()

pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]

if not pdf_files:
    raise ValueError("Please upload at least one PDF file.")

PDF_PATH = pdf_files[0]
print(f"✅ Uploaded PDF: {PDF_PATH}")


Saving Rhea_resume.pdf to Rhea_resume.pdf
✅ Uploaded PDF: Rhea_resume.pdf


## 5. Build FAISS Vector Store

This is the RAG part, similar to your reference notebook:

1. Load PDF pages  
2. Split pages into chunks  
3. Convert chunks to embeddings  
4. Store embeddings in FAISS  
5. Search relevant chunks during question answering


In [5]:
# ============================================================
# 5. Build the RAG vector store
# ============================================================

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Total pages loaded: {len(documents)}")
print(f"Total chunks created: {len(chunks)}")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("✅ FAISS vector store created")


Total pages loaded: 1
Total chunks created: 7


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS vector store created


## 6. Create Tools

The agent gets three tools.

### Tool 1: Search
Use this when the question needs **latest/current information**.

### Tool 2: Weather
Use this when the question asks about **weather in a city**.

### Tool 3: RAG
Use this when the question asks about the **uploaded PDF**.


In [6]:
# ============================================================
# 6A. External Search Tool using SerpAPI
# ============================================================

@tool
def search_tool(query: str) -> str:
    """Search Google for current information using SerpAPI. Use this for latest news, recent facts, current events, or anything that may have changed recently."""
    print("🔎 SERPAPI SEARCH TOOL CALLED")
    serpapi_key = os.environ.get("SERPAPI_API_KEY")
    search = SerpAPIWrapper(serpapi_api_key=serpapi_key)
    return search.run(query)


In [7]:
# ============================================================
# 6B. Weather Tool using OpenWeatherMap API
# ============================================================

@tool
def weather_tool(location: str) -> str:
    """Get current weather for a city or location using OpenWeatherMap. Use this when the user asks about temperature, rain, humidity, wind, or weather conditions."""
    print("🌦️ WEATHER TOOL CALLED")
    api_key = os.environ.get("OPENWEATHER_API_KEY")

    if not api_key:
        return "OPENWEATHER_API_KEY is missing. Please set the API key first."

    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": location,
        "appid": api_key,
        "units": "metric"
    }

    response = requests.get(url, params=params, timeout=20)

    if response.status_code != 200:
        return f"Weather API error: {response.status_code} - {response.text}"

    data = response.json()

    city = data.get("name", location)
    country = data.get("sys", {}).get("country", "")
    weather = data.get("weather", [{}])[0].get("description", "unknown")
    temp = data.get("main", {}).get("temp", "unknown")
    feels_like = data.get("main", {}).get("feels_like", "unknown")
    humidity = data.get("main", {}).get("humidity", "unknown")
    wind_speed = data.get("wind", {}).get("speed", "unknown")

    return (
        f"Current weather in {city}, {country}:\n"
        f"- Condition: {weather}\n"
        f"- Temperature: {temp}°C\n"
        f"- Feels like: {feels_like}°C\n"
        f"- Humidity: {humidity}%\n"
        f"- Wind speed: {wind_speed} m/s"
    )


In [8]:
# ============================================================
# 6C. RAG Tool using FAISS Retriever
# ============================================================

@tool
def rag_tool(question: str) -> str:
    """Search the uploaded PDF and return relevant context. Use this when the user asks about the uploaded document, resume, policy, notes, or internal knowledge base."""
    print("📚 RAG TOOL CALLED")

    docs = retriever.invoke(question)

    if not docs:
        return "No relevant information found in the uploaded PDF."

    formatted_chunks = []
    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page", "unknown")
        source = doc.metadata.get("source", PDF_PATH)
        formatted_chunks.append(
            f"Chunk {i} | Source: {source} | Page: {page}\n{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


## 7. Initialize the LLM

This version uses **OpenRouter** through the dedicated LangChain integration.

You can change the model if needed. For class demos, choose a reliable low-cost model available in your OpenRouter account.


In [9]:
# ============================================================
# 7. LLM using OpenRouter
# ============================================================

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0.2,
    api_key=os.environ["OPENROUTER_API_KEY"],
    max_tokens = 150
)

print("✅ LLM initialized")


✅ LLM initialized


## 8. Create the LangChain Agent

The system prompt tells the agent when to use which tool.

Important teaching point:

> The agent does not blindly call all tools.  
> It decides the right tool based on the user question.


In [10]:
# ============================================================
# 8. Create agent
# ============================================================

tools = [search_tool, weather_tool, rag_tool]

system_prompt = """
You are a helpful AI class-demo assistant.

You have access to three tools:

1. search_tool:
   Use for latest news, current facts, recent information, or external web knowledge.

2. weather_tool:
   Use for current weather, temperature, humidity, wind, or rain questions.

3. rag_tool:
   Use for questions about the uploaded PDF or private document.

Rules:
- If the user asks about the uploaded document, use rag_tool.
- If the user asks about weather, use weather_tool.
- If the user asks about latest/current information, use search_tool.
- If a question needs multiple tools, use multiple tools.
- Always give a clear final answer.
- Mention which tool(s) you used at the end.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

print("✅ Agent created")


✅ Agent created


## 9. Helper Function to Chat with Agent

This makes the demo easier to run.


## 9. Chat History Variations

Below are three simple approaches. Run only the approach you want to demonstrate.

1. **Complete history** – sends the entire conversation every time.
2. **Last 10 messages** – keeps a sliding window to control token usage.
3. **Summary + last 10 messages** – summarizes older messages and keeps recent messages unchanged.

In [11]:
# ============================================================
# 9A. Variation 1 — Keep complete chat history
# ============================================================

full_chat_history = []


def ask_agent_full_history(question: str):
    global full_chat_history

    messages = full_chat_history + [
        {"role": "user", "content": question}
    ]

    result = agent.invoke({"messages": messages})

    full_chat_history = result["messages"]

    answer = full_chat_history[-1].content
    print(f"User: {question}")
    print(f"Assistant: {answer}\n")
    return answer


def clear_full_history():
    global full_chat_history
    full_chat_history = []
    print("✅ Complete chat history cleared")

In [24]:
clear_full_history()

✅ Complete chat history cleared


In [25]:
# Example

ask_agent_full_history("What is the current weather in Delhi?")

🌦️ WEATHER TOOL CALLED
User: What is the current weather in Delhi?
Assistant: The current weather in Delhi is as follows:
- Condition: Overcast clouds
- Temperature: 31.96°C
- Feels like: 38.96°C
- Humidity: 74%
- Wind speed: 1.54 m/s

If you need more information, feel free to ask!



'The current weather in Delhi is as follows:\n- Condition: Overcast clouds\n- Temperature: 31.96°C\n- Feels like: 38.96°C\n- Humidity: 74%\n- Wind speed: 1.54 m/s\n\nIf you need more information, feel free to ask!'

In [26]:

ask_agent_full_history("Based on that, should I carry an umbrella?")

User: Based on that, should I carry an umbrella?
Assistant: Since the weather in Delhi is currently overcast, there is a possibility of rain. It would be a good idea to carry an umbrella just in case. 

If you have any other questions or need further assistance, let me know!



'Since the weather in Delhi is currently overcast, there is a possibility of rain. It would be a good idea to carry an umbrella just in case. \n\nIf you have any other questions or need further assistance, let me know!'

In [28]:
ask_agent_full_history("When is India's Next match?")

🔎 SERPAPI SEARCH TOOL CALLED
User: When is India's Next match?
Assistant: India's next cricket match is scheduled for **October 22, 2023**, against **New Zealand** at the **HPCA Stadium** as part of the ICC Cricket World Cup. 

If you need more details or have any other questions, feel free to ask!



"India's next cricket match is scheduled for **October 22, 2023**, against **New Zealand** at the **HPCA Stadium** as part of the ICC Cricket World Cup. \n\nIf you need more details or have any other questions, feel free to ask!"

In [29]:
full_chat_history

[HumanMessage(content='What is the current weather in Delhi?', additional_kwargs={}, response_metadata={}, id='36990380-1ff9-4b01-8bb5-b19507512503'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1787243119-oATCqhBh17A4kDxZVG5I', 'created': 1787243119, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 5.595e-05, 'cost_details': {'upstream_inference_completions_cost': 8.4e-06, 'upstream_inference_prompt_cost': 4.755e-05, 'upstream_inference_cost': 5.595e-05}, 'system_fingerprint': 'fp_5691373d70'}, id='lc_run--01a01ffd-c3f8-7322-8553-0024c9fc616d-0', tool_calls=[{'name': 'weather_tool', 'args': {'location': 'Delhi'}, 'id': 'call_w2gx1DkMoV8z02hDvVj7Zvjy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 317, 'output_tokens': 14, 'total_tokens': 331, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_d

In [31]:
# ============================================================
# 9B. Variation 2 — Keep only the last 10 messages
# ============================================================

window_chat_history = []
MAX_MESSAGES = 10


def ask_agent_last_10(question: str):
    global window_chat_history

    # Only send the latest 10 stored messages plus the new question.
    recent_messages = window_chat_history[-MAX_MESSAGES:]
    messages = recent_messages + [
        {"role": "user", "content": question}
    ]

    result = agent.invoke({"messages": messages})

    # Keep only the latest 10 messages returned by the agent.
    # Tool calls and tool outputs also count as messages.
    window_chat_history = result["messages"][-MAX_MESSAGES:]

    answer = window_chat_history[-1].content
    print(f"User: {question}")
    print(f"Assistant: {answer}")
    print(f"Messages currently stored: {len(window_chat_history)}\n")
    return answer


def clear_last_10_history():
    global window_chat_history
    window_chat_history = []
    print("✅ Last-10 chat history cleared")

In [ ]:
# Example
clear_last_10_history()

In [32]:

ask_agent_last_10("What is the current weather in Kolkata?")
ask_agent_last_10("What is the humidity?")
ask_agent_last_10("Based on that, should I carry an umbrella?")

🌦️ WEATHER TOOL CALLED
User: What is the current weather in Kolkata?
Assistant: The current weather in Kolkata is as follows:
- Condition: Overcast clouds
- Temperature: 29.97°C
- Feels like: 36.97°C
- Humidity: 89%
- Wind speed: 3.6 m/s

If you need more information, feel free to ask!
Messages currently stored: 4

User: What is the humidity?
Assistant: The humidity in Kolkata is currently 89%.
Messages currently stored: 6

User: Based on that, should I carry an umbrella?
Assistant: Given the high humidity of 89% and the overcast clouds, it's advisable to carry an umbrella, as there is a possibility of rain. It's better to be prepared for wet weather conditions.
Messages currently stored: 8



"Given the high humidity of 89% and the overcast clouds, it's advisable to carry an umbrella, as there is a possibility of rain. It's better to be prepared for wet weather conditions."

In [33]:
# ============================================================
# 9C. Variation 3 — Summarize older history and keep last 10 messages
# ============================================================

summary_text = ""
summary_chat_history = []
SUMMARY_WINDOW = 10


def message_to_text(message):
    """Convert LangChain or dictionary messages into readable text."""
    role = getattr(message, "type", None) or getattr(message, "role", None)
    content = getattr(message, "content", None)

    if content is None and isinstance(message, dict):
        role = message.get("role", "message")
        content = message.get("content", "")

    return f"{role}: {content}"


def update_conversation_summary(messages_to_summarize):
    global summary_text

    conversation_text = "\n".join(
        message_to_text(message) for message in messages_to_summarize
    )

    summary_prompt = f"""
You maintain a concise conversation memory.

Existing summary:
{summary_text or 'No previous summary.'}

New conversation messages:
{conversation_text}

Create an updated concise summary. Preserve important facts, user preferences,
questions, tool findings, and unresolved tasks. Do not add new information.
"""

    response = llm.invoke(summary_prompt)
    summary_text = response.content


def ask_agent_summary_last_10(question: str):
    global summary_chat_history, summary_text

    # When more than 10 recent messages exist, summarize the older messages.
    if len(summary_chat_history) > SUMMARY_WINDOW:
        older_messages = summary_chat_history[:-SUMMARY_WINDOW]
        update_conversation_summary(older_messages)
        summary_chat_history = summary_chat_history[-SUMMARY_WINDOW:]

    messages = []

    if summary_text:
        messages.append({
            "role": "system",
            "content": f"Conversation summary from earlier messages:\n{summary_text}"
        })

    messages.extend(summary_chat_history[-SUMMARY_WINDOW:])
    messages.append({"role": "user", "content": question})

    result = agent.invoke({"messages": messages})

    # Remove the temporary summary system message before storing recent history.
    returned_messages = result["messages"]
    if summary_text and returned_messages:
        returned_messages = returned_messages[1:]

    summary_chat_history = returned_messages

    # Summarize immediately if this interaction pushed the history over the limit.
    if len(summary_chat_history) > SUMMARY_WINDOW:
        older_messages = summary_chat_history[:-SUMMARY_WINDOW]
        update_conversation_summary(older_messages)
        summary_chat_history = summary_chat_history[-SUMMARY_WINDOW:]

    answer = summary_chat_history[-1].content
    print(f"User: {question}")
    print(f"Assistant: {answer}")
    print(f"Recent messages stored: {len(summary_chat_history)}")
    print(f"Summary available: {bool(summary_text)}\n")
    return answer


def clear_summary_history():
    global summary_text, summary_chat_history
    summary_text = ""
    summary_chat_history = []
    print("✅ Summary and recent chat history cleared")

In [34]:
# Example
clear_summary_history()

✅ Summary and recent chat history cleared


In [35]:

ask_agent_summary_last_10("What is the current weather in Kolkata?")
ask_agent_summary_last_10("What is the humidity?")
ask_agent_summary_last_10("Remember that I prefer short answers.")
ask_agent_summary_last_10("Based on the weather, should I carry an umbrella?")

🌦️ WEATHER TOOL CALLED
User: What is the current weather in Kolkata?
Assistant: The current weather in Kolkata is as follows:
- Condition: Overcast clouds
- Temperature: 29.97°C
- Feels like: 36.97°C
- Humidity: 89%
- Wind speed: 3.6 m/s

If you need more information, feel free to ask!
Recent messages stored: 4
Summary available: False

User: What is the humidity?
Assistant: The humidity in Kolkata is currently 89%.
Recent messages stored: 6
Summary available: False

User: Remember that I prefer short answers.
Assistant: Got it! The humidity in Kolkata is 89%.
Recent messages stored: 8
Summary available: False

User: Based on the weather, should I carry an umbrella?
Assistant: Yes, since the weather is overcast and the humidity is high, it's a good idea to carry an umbrella.
Recent messages stored: 10
Summary available: False



"Yes, since the weather is overcast and the humidity is high, it's a good idea to carry an umbrella."

In [36]:
ask_agent_summary_last_10("Will i get drenched if i go outside?")

User: Will i get drenched if i go outside?
Assistant: While the current weather is overcast, it doesn't specify rain. However, with high humidity, there's a possibility of rain. It's best to be prepared for potential rain to avoid getting drenched.
Recent messages stored: 10
Summary available: True



"While the current weather is overcast, it doesn't specify rain. However, with high humidity, there's a possibility of rain. It's best to be prepared for potential rain to avoid getting drenched."

In [ ]:
summary_text

'Updated summary:\nUser asked about the current weather in Kolkata.'

### Important note

A single agent interaction can create several messages: user message, assistant tool call, tool result, and final assistant answer. Therefore, **10 messages does not always mean 10 user questions**. For a classroom demo, this is useful because it shows how agent/tool history actually works.